# Face Mask Detection From Scratch - Colab Runner

Notebook ini disiapkan untuk Google Colab. Alurnya tetap sama dengan runner utama: EDA, preprocessing, model klasik HOG-SVM, CNN from scratch, evaluasi, dan ekspor hasil akhir.


In [ ]:
# Colab bootstrap: jalankan cell ini sekali di awal.
import os
from pathlib import Path

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    try:
        import google.colab  # noqa: F401
        print('Running on Google Colab')
    except Exception:
        print('Running with /content path available')

    # Kaggle API biasanya belum tersedia di runtime Colab yang bersih.
    try:
        import kaggle  # noqa: F401
    except Exception:
        !pip -q install kaggle kagglehub

    kaggle_json = Path('/root/.kaggle/kaggle.json')
    if not kaggle_json.exists():
        print('Upload kaggle.json dari akun Kaggle kamu untuk download dataset.')
        try:
            from google.colab import files
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                kaggle_json.parent.mkdir(parents=True, exist_ok=True)
                Path('kaggle.json').replace(kaggle_json)
                kaggle_json.chmod(0o600)
                print('kaggle.json tersimpan di /root/.kaggle/kaggle.json')
        except Exception as error:
            print('Upload kaggle.json dilewati:', repr(error))


# 01_eda.ipynb


# Data Setup, EDA, dan Split

Notebook ini menyiapkan konfigurasi, memuat dataset, menjalankan EDA pada gambar asli, lalu membuat split data di cell paling akhir.

# Face Mask Detection: Full CV Pipeline From Scratch

Notebook ini membangun pipeline computer vision lengkap untuk klasifikasi penggunaan masker wajah. Seluruh model deep learning dibuat manual menggunakan layer dasar `Conv2D` dan dilatih dari awal tanpa bobot eksternal.

## 1. Desain Eksperimen

Dataset yang digunakan adalah Face Mask Dataset dari Kaggle dengan dua kelas utama: `with_mask` dan `without_mask`. Split dibuat sekali di level file asli dengan rasio 70/15/15 untuk train, validation, dan test. Semua skenario memakai split yang sama agar tidak terjadi data leakage.

Pipeline akademik yang diuji:

- preprocessing: resize 128x128, normalisasi 0-1, label binary
- enhancement/restoration: CLAHE dan Gaussian denoise ringan
- face segmentation/cropping: Haar Cascade dengan fallback ke full image
- feature extraction klasik: HOG
- classification klasik: SVM
- classification deep learning: custom CNN Conv2D from scratch dengan opsi SE block
- evaluasi: accuracy, precision, recall, F1-score, classification report, confusion matrix, ROC curve, AUC, dan kurva training

Skenario wajib:

- HOG+SVM vs CNN
- enhancement vs tanpa enhancement
- crop vs full image
- augmentation vs tanpa augmentation
- SE block vs tanpa SE block

Catatan penting: notebook ini tidak memakai model siap pakai, metode pemindahan bobot, atau bobot eksternal. Arsitektur CNN dibuat manual dari layer `Conv2D`.

## 2. Setup

Konfigurasi dibuat eksplisit agar eksperimen mudah direproduksi dan mudah dijelaskan saat presentasi/laporan.

In [ ]:
import json
import os
import random
import shutil
import subprocess
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED = 42
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 64
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 5
LR_PATIENCE = 2

RUN_OPTIONAL_CROP_SCENARIO = True
RUN_SE_ABLATION = True
RUN_LIGHT_TUNING = True

CLASS_NAMES = ['with_mask', 'without_mask']
CLASS_TO_LABEL = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

OUTPUT_DIR = Path('/content/results')
TEMP_DIR = Path('/content/temp')
DEFAULT_DATASET_DIR = Path('/content/datasets/face-mask-dataset')
SCENARIO_ROOT = TEMP_DIR / 'face_mask_scenarios_final'
DATASET_HANDLE = 'omkargurav/face-mask-dataset'

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print('Physical GPUs:', gpus)
if gpus:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())
    except Exception as error:
        print('Mixed precision not enabled:', repr(error))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('Logical GPUs:', tf.config.list_logical_devices('GPU'))

## 3. Load Dataset Kaggle

Notebook mencari folder dataset yang berisi class `with_mask` dan `without_mask`. Jika dataset belum ter-mount di Kaggle, fallback download disediakan untuk menjaga notebook tetap reproducible.

In [ ]:
def print_tree(root: Path, max_depth: int = 3, limit: int = 120) -> None:
    print(f'Dataset tree preview: {root}')
    if not root.exists():
        print('Dataset path does not exist.')
        return

    root_depth = len(root.parts)
    shown = 0
    for path in sorted(root.rglob('*')):
        depth = len(path.parts) - root_depth
        if depth > max_depth:
            continue
        indent = '  ' * max(depth - 1, 0)
        suffix = '/' if path.is_dir() else ''
        print(f'{indent}{path.name}{suffix}')
        shown += 1
        if shown >= limit:
            print('... tree preview truncated ...')
            break


def find_image_directory(dataset_root: Path) -> Path:
    directories = [dataset_root]
    if dataset_root.exists():
        directories.extend([p for p in dataset_root.rglob('*') if p.is_dir()])

    for directory in directories:
        child_dirs = {p.name.lower(): p for p in directory.iterdir() if p.is_dir()}
        if all(name in child_dirs for name in CLASS_NAMES):
            return directory

    raise FileNotFoundError(f'Could not find class folders {CLASS_NAMES} under {dataset_root}')


def find_mounted_dataset() -> Path | None:
    if DEFAULT_DATASET_DIR.exists():
        return DEFAULT_DATASET_DIR

    colab_dataset_root = Path('/content/datasets')
    if colab_dataset_root.exists():
        for pattern in ['*face*mask*', '*mask*', '*face*']:
            matches = [p for p in colab_dataset_root.glob(pattern) if p.is_dir()]
            if matches:
                return matches[0]

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        print('Available /kaggle/input entries:', [p.name for p in kaggle_input.iterdir()])
        for pattern in ['*face*mask*', '*mask*', '*face*']:
            matches = [p for p in kaggle_input.glob(pattern) if p.is_dir()]
            if matches:
                return matches[0]
    return None


def download_dataset_fallback() -> Path:
    download_dir = Path('/content/datasets/face-mask-dataset')
    download_dir.mkdir(parents=True, exist_ok=True)

    try:
        import kagglehub

        print('Dataset mount not found. Trying kagglehub fallback download...')
        downloaded_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
        print('kagglehub downloaded dataset to:', downloaded_path)
        return downloaded_path
    except Exception as error:
        print('kagglehub fallback failed:', repr(error))

    print('Trying Kaggle CLI fallback download...')
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', DATASET_HANDLE, '-p', str(download_dir), '--unzip'],
        check=True,
    )
    return download_dir


dataset_root = find_mounted_dataset()
if dataset_root is None:
    dataset_root = download_dataset_fallback()

print_tree(dataset_root)
original_image_dir = find_image_directory(dataset_root)
print('Selected image directory:', original_image_dir)

## 4. Inventaris Dataset Asli

Path gambar dikumpulkan per class tanpa mengubah citra. Inventaris ini menjadi dasar semua analisis sebelum preprocessing dan split data.

In [ ]:

def collect_image_paths(image_dir: Path) -> dict[str, list[Path]]:
    paths_by_class = {}
    for class_name in CLASS_NAMES:
        class_dir = image_dir / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f'Missing class folder: {class_dir}')
        paths = [p for p in sorted(class_dir.rglob('*')) if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        if not paths:
            raise ValueError(f'No images found for class: {class_name}')
        paths_by_class[class_name] = paths
    return paths_by_class


def load_original_rgb(image_path: Path) -> np.ndarray | None:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return None
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


paths_by_class = collect_image_paths(original_image_dir)
class_counts = {class_name: len(paths) for class_name, paths in paths_by_class.items()}
print('Class counts:', class_counts)
print('Total images:', sum(class_counts.values()))


## 5. Distribusi Kelas

Jumlah gambar pada kelas `with_mask` dan `without_mask` dihitung lalu divisualisasikan dalam bar chart. Analisis ini bertujuan melihat keseimbangan data sejak awal, karena komposisi kelas akan memengaruhi cara membaca performa akhir seperti accuracy, precision, recall, F1-score, dan AUC.


In [ ]:

class_values = [class_counts[class_name] for class_name in CLASS_NAMES]

plt.figure(figsize=(7, 4))
bars = plt.bar(CLASS_NAMES, class_values, color=['#2f80ed', '#27ae60'])
plt.title('Distribusi Class Dataset Asli')
plt.xlabel('Class')
plt.ylabel('Jumlah gambar')
plt.bar_label(bars)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_class_distribution.png', dpi=160)
plt.show()


## 6. Visualisasi Sampel Gambar

Beberapa gambar asli dari setiap kelas ditampilkan secara acak tanpa preprocessing. Alurnya dimulai dari mengambil sampel per kelas, menampilkan citra dalam grid, lalu mengamati kesesuaian label, variasi bentuk masker, ekspresi wajah, jarak kamera, pencahayaan, dan kondisi background.


In [ ]:

rng = random.Random(SEED)
fig, axes = plt.subplots(len(CLASS_NAMES), 6, figsize=(13, 5))
for row_idx, class_name in enumerate(CLASS_NAMES):
    sample_paths = rng.sample(paths_by_class[class_name], k=min(6, len(paths_by_class[class_name])))
    for col_idx, image_path in enumerate(sample_paths):
        image_rgb = load_original_rgb(image_path)
        axes[row_idx, col_idx].imshow(image_rgb)
        axes[row_idx, col_idx].set_title(class_name, fontsize=9)
        axes[row_idx, col_idx].axis('off')
plt.suptitle('Contoh Gambar Asli Sebelum Preprocessing', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_original_samples.png', dpi=160, bbox_inches='tight')
plt.show()


## 7. Distribusi Resolusi Gambar

Lebar, tinggi, dan rasio aspek setiap gambar asli dihitung sebelum ukuran gambar diseragamkan. Analisis ini bertujuan memahami variasi dimensi dataset, melihat apakah ada gambar yang terlalu kecil atau sangat tidak proporsional, dan menentukan konsekuensi resize terhadap detail wajah serta masker.


In [ ]:
def collect_eda_image_stats(paths_by_class: dict[str, list[Path]]) -> tuple[list[dict], list[str]]:
    stats = []
    read_failed = []
    for class_name in CLASS_NAMES:
        for image_path in paths_by_class[class_name]:
            image_rgb = load_original_rgb(image_path)
            if image_rgb is None:
                read_failed.append(str(image_path))
                continue
            gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
            height, width = gray.shape
            stats.append(
                {
                    'path': str(image_path),
                    'class_name': class_name,
                    'width': int(width),
                    'height': int(height),
                    'aspect_ratio': float(width / height),
                    'brightness_mean': float(gray.mean()),
                    'contrast_std': float(gray.std()),
                }
            )
    return stats, read_failed


eda_image_stats, eda_read_failed = collect_eda_image_stats(paths_by_class)
print('EDA image count:', len(eda_image_stats))
print('EDA read failed:', len(eda_read_failed))

widths = np.array([item['width'] for item in eda_image_stats])
heights = np.array([item['height'] for item in eda_image_stats])
aspect_ratios = np.array([item['aspect_ratio'] for item in eda_image_stats])
class_labels_for_stats = np.array([item['class_name'] for item in eda_image_stats])

colors = ['#2f80ed', '#f2994a']
fig, axes = plt.subplots(2, len(CLASS_NAMES), figsize=(6 * len(CLASS_NAMES), 7), squeeze=False)

for col_idx, (class_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    class_mask = class_labels_for_stats == class_name
    class_widths = widths[class_mask]
    class_heights = heights[class_mask]
    class_aspect_ratios = aspect_ratios[class_mask]

    axes[0, col_idx].scatter(class_widths, class_heights, alpha=0.25, s=12, color=color)
    axes[0, col_idx].axvline(IMAGE_SIZE[0], color='black', linestyle='--', linewidth=1, label='target width')
    axes[0, col_idx].axhline(IMAGE_SIZE[1], color='gray', linestyle='--', linewidth=1, label='target height')
    axes[0, col_idx].set_title(f'Resolusi - {class_name}')
    axes[0, col_idx].set_xlabel('Width')
    axes[0, col_idx].set_ylabel('Height')
    axes[0, col_idx].legend()

    axes[1, col_idx].hist(class_aspect_ratios, bins=30, color=color, alpha=0.85, edgecolor='white')
    axes[1, col_idx].axvline(1.0, color='black', linestyle='--', linewidth=1, label='rasio 1:1')
    axes[1, col_idx].axvline(np.median(class_aspect_ratios), color=color, linestyle=':', linewidth=1.8, label='median')
    axes[1, col_idx].set_title(f'Aspect Ratio - {class_name}')
    axes[1, col_idx].set_xlabel('Width / Height')
    axes[1, col_idx].set_ylabel('Jumlah gambar')
    axes[1, col_idx].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_resolution_aspect_ratio.png', dpi=160)
plt.show()

eda_summary = {
    'image_count': int(len(eda_image_stats)),
    'read_failed_count': int(len(eda_read_failed)),
    'class_counts': class_counts,
    'width': {'min': int(widths.min()), 'median': float(np.median(widths)), 'max': int(widths.max())},
    'height': {'min': int(heights.min()), 'median': float(np.median(heights)), 'max': int(heights.max())},
    'aspect_ratio': {'min': float(aspect_ratios.min()), 'median': float(np.median(aspect_ratios)), 'max': float(aspect_ratios.max())},
    'figures': {
        'class_distribution': str(OUTPUT_DIR / 'eda_class_distribution.png'),
        'original_samples': str(OUTPUT_DIR / 'eda_original_samples.png'),
        'resolution_aspect_ratio': str(OUTPUT_DIR / 'eda_resolution_aspect_ratio.png'),
    },
}


## 8. Analisis Brightness dan Kontras

Setiap gambar dikonversi ke grayscale, lalu dihitung rata-rata intensitas sebagai brightness dan standar deviasi intensitas sebagai kontras. Distribusi nilai per kelas digunakan untuk membaca variasi pencahayaan dan ketajaman intensitas lokal, terutama apakah dataset banyak berisi gambar terlalu gelap, terlalu terang, atau rendah kontras.


In [ ]:
brightness = np.array([item['brightness_mean'] for item in eda_image_stats])
contrast = np.array([item['contrast_std'] for item in eda_image_stats])
brightness_by_class = [brightness[class_labels_for_stats == class_name] for class_name in CLASS_NAMES]
contrast_by_class = [contrast[class_labels_for_stats == class_name] for class_name in CLASS_NAMES]

colors = ['#2f80ed', '#f2994a']
fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(12, 4 * len(CLASS_NAMES)), squeeze=False)

for row_idx, (class_name, color, brightness_values, contrast_values) in enumerate(
    zip(CLASS_NAMES, colors, brightness_by_class, contrast_by_class)
):
    axes[row_idx, 0].hist(brightness_values, bins=30, alpha=0.85, color=color, edgecolor='white')
    axes[row_idx, 0].axvline(np.median(brightness_values), color='black', linestyle='--', linewidth=1.6, label='median')
    axes[row_idx, 0].set_title(f'Brightness - {class_name}')
    axes[row_idx, 0].set_xlabel('Mean grayscale')
    axes[row_idx, 0].set_ylabel('Jumlah gambar')
    axes[row_idx, 0].legend()

    axes[row_idx, 1].hist(contrast_values, bins=30, alpha=0.85, color=color, edgecolor='white')
    axes[row_idx, 1].axvline(np.median(contrast_values), color='black', linestyle='--', linewidth=1.6, label='median')
    axes[row_idx, 1].set_title(f'Kontras - {class_name}')
    axes[row_idx, 1].set_xlabel('Std grayscale')
    axes[row_idx, 1].set_ylabel('Jumlah gambar')
    axes[row_idx, 1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_brightness_contrast.png', dpi=160)
plt.show()

eda_summary['brightness_mean'] = {'min': float(brightness.min()), 'median': float(np.median(brightness)), 'max': float(brightness.max())}
eda_summary['contrast_std'] = {'min': float(contrast.min()), 'median': float(np.median(contrast)), 'max': float(contrast.max())}
eda_summary['figures']['brightness_contrast'] = str(OUTPUT_DIR / 'eda_brightness_contrast.png')


## 9. Analisis Noise dan Blur

Blur dihitung menggunakan variance of Laplacian, sedangkan noise diperkirakan dari standar deviasi residual antara gambar grayscale dan hasil Gaussian blur ringan. Analisis ini bertujuan melihat kualitas citra asli, membedakan gambar yang detailnya tajam dari gambar yang kabur, dan mengetahui seberapa besar gangguan tekstur halus pada dataset.


In [ ]:
def estimate_noise_blur(paths_by_class: dict[str, list[Path]]) -> list[dict]:
    records = []
    for class_name in CLASS_NAMES:
        for image_path in paths_by_class[class_name]:
            image_rgb = load_original_rgb(image_path)
            if image_rgb is None:
                continue
            gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
            blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
            smooth = cv2.GaussianBlur(gray, (3, 3), sigmaX=0.5)
            noise_score = float(np.std(gray.astype(np.float32) - smooth.astype(np.float32)))
            records.append({'class_name': class_name, 'blur_score': blur_score, 'noise_score': noise_score})
    return records


noise_blur_records = estimate_noise_blur(paths_by_class)
blur_scores = np.array([item['blur_score'] for item in noise_blur_records])
noise_scores = np.array([item['noise_score'] for item in noise_blur_records])
noise_blur_labels = np.array([item['class_name'] for item in noise_blur_records])

blur_by_class = [blur_scores[noise_blur_labels == class_name] for class_name in CLASS_NAMES]
noise_by_class = [noise_scores[noise_blur_labels == class_name] for class_name in CLASS_NAMES]
blur_log_by_class = [np.log1p(values) for values in blur_by_class]

colors = ['#2f80ed', '#f2994a']
fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(12, 4 * len(CLASS_NAMES)), squeeze=False)

for row_idx, (class_name, color, blur_values, noise_values) in enumerate(
    zip(CLASS_NAMES, colors, blur_log_by_class, noise_by_class)
):
    axes[row_idx, 0].hist(blur_values, bins=30, alpha=0.85, color=color, edgecolor='white')
    axes[row_idx, 0].axvline(np.median(blur_values), color='black', linestyle='--', linewidth=1.6, label='median')
    axes[row_idx, 0].set_title(f'Blur - {class_name}')
    axes[row_idx, 0].set_xlabel('log1p variance of Laplacian')
    axes[row_idx, 0].set_ylabel('Jumlah gambar')
    axes[row_idx, 0].legend()

    axes[row_idx, 1].hist(noise_values, bins=30, alpha=0.85, color=color, edgecolor='white')
    axes[row_idx, 1].axvline(np.median(noise_values), color='black', linestyle='--', linewidth=1.6, label='median')
    axes[row_idx, 1].set_title(f'Noise - {class_name}')
    axes[row_idx, 1].set_xlabel('Std residual')
    axes[row_idx, 1].set_ylabel('Jumlah gambar')
    axes[row_idx, 1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_noise_blur.png', dpi=160)
plt.show()

eda_summary['blur_score'] = {'min': float(blur_scores.min()), 'median': float(np.median(blur_scores)), 'max': float(blur_scores.max())}
eda_summary['noise_score'] = {'min': float(noise_scores.min()), 'median': float(np.median(noise_scores)), 'max': float(noise_scores.max())}
eda_summary['figures']['noise_blur'] = str(OUTPUT_DIR / 'eda_noise_blur.png')


## 10. Visualisasi Edge dan HOG

Sampel gambar ditampilkan bersama Canny edge dan visualisasi orientasi gradien. Alur ini membantu mengamati struktur tepi yang muncul pada area wajah, masker, hidung, mulut, dan kontur kepala, sehingga pola bentuk yang dominan pada citra dapat terlihat sebelum data masuk ke tahap pemodelan.


In [ ]:

def draw_hog_orientation(image_rgb: np.ndarray, cell_size: int = 8, bins: int = 9) -> np.ndarray:
    resized = cv2.resize(image_rgb, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=1)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=1)
    magnitude, angle = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    angle = angle % 180

    canvas = np.zeros_like(gray, dtype=np.uint8)
    bin_width = 180 / bins
    for y in range(0, gray.shape[0], cell_size):
        for x in range(0, gray.shape[1], cell_size):
            mag_cell = magnitude[y:y + cell_size, x:x + cell_size]
            ang_cell = angle[y:y + cell_size, x:x + cell_size]
            if mag_cell.size == 0:
                continue
            hist = np.zeros(bins, dtype=np.float32)
            bin_ids = np.minimum((ang_cell / bin_width).astype(int), bins - 1)
            for bin_id in range(bins):
                hist[bin_id] = mag_cell[bin_ids == bin_id].sum()
            if hist.max() <= 0:
                continue
            dominant = int(hist.argmax())
            theta = np.deg2rad((dominant + 0.5) * bin_width)
            length = int((hist[dominant] / (hist.max() + 1e-6)) * cell_size * 0.45)
            cx = x + cell_size // 2
            cy = y + cell_size // 2
            dx = int(np.cos(theta) * length)
            dy = int(np.sin(theta) * length)
            cv2.line(canvas, (cx - dx, cy - dy), (cx + dx, cy + dy), 255, 1)
    return canvas


fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(9, 6))
rng = random.Random(SEED + 7)
for row_idx, class_name in enumerate(CLASS_NAMES):
    image_path = rng.choice(paths_by_class[class_name])
    image_rgb = load_original_rgb(image_path)
    resized = cv2.resize(image_rgb, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, threshold1=80, threshold2=160)
    hog_view = draw_hog_orientation(image_rgb)

    axes[row_idx, 0].imshow(resized)
    axes[row_idx, 0].set_title(f'Original - {class_name}', fontsize=9)
    axes[row_idx, 1].imshow(edges, cmap='gray')
    axes[row_idx, 1].set_title('Canny Edge', fontsize=9)
    axes[row_idx, 2].imshow(hog_view, cmap='gray')
    axes[row_idx, 2].set_title('HOG Orientation', fontsize=9)
    for col_idx in range(3):
        axes[row_idx, col_idx].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_edge_hog_examples.png', dpi=160)
plt.show()

eda_summary['figures']['edge_hog_examples'] = str(OUTPUT_DIR / 'eda_edge_hog_examples.png')


## 11. Analisis Variasi Pose dan Background

Deteksi wajah digunakan untuk memperkirakan area wajah, posisi wajah terhadap pusat gambar, serta variasi warna dan brightness pada area tepi sebagai gambaran background. Analisis ini bertujuan membaca keragaman pose, skala wajah, framing, dan kondisi latar yang dapat membuat gambar antar sampel berbeda walaupun label kelasnya sama.


In [ ]:
cascade_path = Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'
eda_face_detector = cv2.CascadeClassifier(str(cascade_path))
face_detection_records = []
face_examples_detected = []
face_examples_failed = []

for item in eda_image_stats:
    image_path = Path(item['path'])
    image_rgb = load_original_rgb(image_path)
    height, width = image_rgb.shape[:2]
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    faces = eda_face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(35, 35))
    detected = len(faces) > 0

    border = max(4, int(0.08 * min(height, width)))
    border_pixels = np.concatenate(
        [
            image_rgb[:border, :, :].reshape(-1, 3),
            image_rgb[-border:, :, :].reshape(-1, 3),
            image_rgb[:, :border, :].reshape(-1, 3),
            image_rgb[:, -border:, :].reshape(-1, 3),
        ],
        axis=0,
    )
    background_color_std = float(border_pixels.std(axis=0).mean())
    background_brightness = float(cv2.cvtColor(border_pixels.reshape(-1, 1, 3), cv2.COLOR_RGB2GRAY).mean())

    face_area_ratio = None
    face_center_offset = None
    if detected:
        x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
        face_area_ratio = float((w * h) / (width * height))
        face_cx = x + w / 2
        face_cy = y + h / 2
        face_center_offset = float(np.sqrt(((face_cx / width) - 0.5) ** 2 + ((face_cy / height) - 0.5) ** 2))
        if len(face_examples_detected) < 4:
            preview = image_rgb.copy()
            cv2.rectangle(preview, (x, y), (x + w, y + h), (255, 60, 60), 3)
            face_examples_detected.append((preview, item['class_name']))
    elif len(face_examples_failed) < 4:
        face_examples_failed.append((image_rgb, item['class_name']))

    face_detection_records.append(
        {
            **item,
            'face_detected': bool(detected),
            'face_area_ratio': face_area_ratio,
            'face_center_offset': face_center_offset,
            'background_color_std': background_color_std,
            'background_brightness': background_brightness,
        }
    )

face_area_ratios = np.array([record['face_area_ratio'] for record in face_detection_records if record['face_area_ratio'] is not None])
face_center_offsets = np.array([record['face_center_offset'] for record in face_detection_records if record['face_center_offset'] is not None])
background_color_std = np.array([record['background_color_std'] for record in face_detection_records])
background_brightness = np.array([record['background_brightness'] for record in face_detection_records])

pose_metrics = [
    ('face_area_ratio', 'Skala Wajah Terdeteksi', 'Face area / image area', '#2f80ed'),
    ('face_center_offset', 'Offset Posisi Wajah dari Tengah', 'Normalized center offset', '#27ae60'),
    ('background_color_std', 'Variasi Warna Background', 'Mean RGB std pada border', '#f2994a'),
    ('background_brightness', 'Brightness Background', 'Mean grayscale pada border', '#8856a7'),
]

fig, axes = plt.subplots(len(pose_metrics), len(CLASS_NAMES), figsize=(6 * len(CLASS_NAMES), 3.5 * len(pose_metrics)), squeeze=False)

for row_idx, (metric_key, title, xlabel, color) in enumerate(pose_metrics):
    for col_idx, class_name in enumerate(CLASS_NAMES):
        values = np.array(
            [
                record[metric_key]
                for record in face_detection_records
                if record['class_name'] == class_name and record[metric_key] is not None
            ]
        )
        axes[row_idx, col_idx].hist(values, bins=30, color=color, alpha=0.85, edgecolor='white')
        if len(values):
            axes[row_idx, col_idx].axvline(np.median(values), color='black', linestyle='--', linewidth=1.4, label='median')
            axes[row_idx, col_idx].legend()
        axes[row_idx, col_idx].set_title(f'{title} - {class_name}')
        axes[row_idx, col_idx].set_xlabel(xlabel)
        axes[row_idx, col_idx].set_ylabel('Jumlah gambar')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_pose_background_variation.png', dpi=160)
plt.show()

eda_summary['pose_background'] = {
    'face_area_ratio_median': float(np.median(face_area_ratios)) if len(face_area_ratios) else None,
    'face_center_offset_median': float(np.median(face_center_offsets)) if len(face_center_offsets) else None,
    'background_color_std_median': float(np.median(background_color_std)),
    'background_brightness_median': float(np.median(background_brightness)),
}
eda_summary['figures']['pose_background_variation'] = str(OUTPUT_DIR / 'eda_pose_background_variation.png')


## 12. Coverage Face Detection Haar Cascade

Haar Cascade dijalankan pada gambar asli untuk menghitung berapa banyak wajah yang berhasil terdeteksi dan menampilkan contoh keberhasilan maupun kegagalannya. Analisis ini bertujuan mengetahui cakupan deteksi wajah pada dataset, termasuk kondisi yang rawan gagal seperti pose miring, wajah kecil, blur, pencahayaan ekstrem, atau background yang ramai.


In [ ]:

face_detection_summary = {}
for class_name in CLASS_NAMES:
    records = [record for record in face_detection_records if record['class_name'] == class_name]
    detected_count = sum(record['face_detected'] for record in records)
    face_detection_summary[class_name] = {
        'total': int(len(records)),
        'detected': int(detected_count),
        'not_detected': int(len(records) - detected_count),
        'detection_rate': float(detected_count / len(records)) if records else 0.0,
    }

all_detected_count = sum(record['face_detected'] for record in face_detection_records)
face_detection_summary['overall'] = {
    'total': int(len(face_detection_records)),
    'detected': int(all_detected_count),
    'not_detected': int(len(face_detection_records) - all_detected_count),
    'detection_rate': float(all_detected_count / len(face_detection_records)) if face_detection_records else 0.0,
}
print('Face detection summary:', json.dumps(face_detection_summary, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
detection_rates = [face_detection_summary[class_name]['detection_rate'] * 100 for class_name in CLASS_NAMES]
bars = axes[0].bar(CLASS_NAMES, detection_rates, color=['#2f80ed', '#27ae60'])
axes[0].set_ylim(0, 100)
axes[0].set_title('Coverage Haar Cascade per Class')
axes[0].set_ylabel('Detection rate (%)')
axes[0].bar_label(bars, fmt='%.1f%%')

overall_values = [face_detection_summary['overall']['detected'], face_detection_summary['overall']['not_detected']]
axes[1].pie(overall_values, labels=['Detected', 'Not detected'], autopct='%1.1f%%', colors=['#27ae60', '#d9534f'])
axes[1].set_title('Coverage Haar Cascade Keseluruhan')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_face_detection_coverage.png', dpi=160)
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, (image_rgb, class_name) in enumerate(face_examples_detected[:4]):
    axes[0, idx].imshow(image_rgb)
    axes[0, idx].set_title(f'Detected - {class_name}', fontsize=9)
    axes[0, idx].axis('off')
for idx, (image_rgb, class_name) in enumerate(face_examples_failed[:4]):
    axes[1, idx].imshow(image_rgb)
    axes[1, idx].set_title(f'Not detected - {class_name}', fontsize=9)
    axes[1, idx].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_face_detection_examples.png', dpi=160)
plt.show()

eda_summary['face_detection_summary'] = face_detection_summary
eda_summary['figures'].update(
    {
        'face_detection_coverage': str(OUTPUT_DIR / 'eda_face_detection_coverage.png'),
        'face_detection_examples': str(OUTPUT_DIR / 'eda_face_detection_examples.png'),
    }
)


## 13. Split 70/15/15 Tanpa Data Leakage

Setelah seluruh analisis gambar asli selesai, daftar file dibagi menjadi train, validation, dan test dengan rasio 70/15/15 secara stratified. Tujuannya memastikan satu file asli hanya masuk ke satu split, menjaga distribusi kelas tetap proporsional, dan membuat seluruh tahap berikutnya memakai pembagian data yang sama.


In [ ]:

def split_paths(paths_by_class: dict[str, list[Path]]) -> dict[str, list[tuple[Path, str]]]:
    rng = random.Random(SEED)
    splits = {'train': [], 'validation': [], 'test': []}

    for class_name, paths in paths_by_class.items():
        shuffled = paths.copy()
        rng.shuffle(shuffled)
        n_total = len(shuffled)
        n_train = int(n_total * TRAIN_RATIO)
        n_val = int(n_total * VAL_RATIO)

        splits['train'].extend((path, class_name) for path in shuffled[:n_train])
        splits['validation'].extend((path, class_name) for path in shuffled[n_train:n_train + n_val])
        splits['test'].extend((path, class_name) for path in shuffled[n_train + n_val:])

    for split_name in splits:
        rng.shuffle(splits[split_name])
    return splits


def count_split_labels(splits: dict[str, list[tuple[Path, str]]]) -> dict[str, dict[str, int]]:
    counts = {}
    for split_name, records in splits.items():
        counts[split_name] = {class_name: 0 for class_name in CLASS_NAMES}
        for _, class_name in records:
            counts[split_name][class_name] += 1
    return counts


splits = split_paths(paths_by_class)
split_counts = count_split_labels(splits)
split_sizes = {split_name: len(records) for split_name, records in splits.items()}

all_split_paths = []
for records in splits.values():
    all_split_paths.extend(str(path.resolve()) for path, _ in records)
assert len(all_split_paths) == len(set(all_split_paths)), 'Data leakage detected: duplicated original path across splits.'

print('Split sizes:', split_sizes)
print('Split label counts:', split_counts)

split_names = list(splits.keys())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

class_values = [class_counts[class_name] for class_name in CLASS_NAMES]
bars = axes[0].bar(CLASS_NAMES, class_values, color=['#2f80ed', '#27ae60'])
axes[0].set_title('Distribusi Class Dataset Asli')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Jumlah gambar')
axes[0].bar_label(bars)

bottom = np.zeros(len(split_names))
for class_name, color in zip(CLASS_NAMES, ['#2f80ed', '#27ae60']):
    values = [split_counts[split_name][class_name] for split_name in split_names]
    axes[1].bar(split_names, values, bottom=bottom, label=class_name, color=color)
    bottom += np.array(values)
axes[1].set_title('Distribusi Label per Split')
axes[1].set_xlabel('Split')
axes[1].set_ylabel('Jumlah gambar')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_class_split_distribution.png', dpi=160)
plt.show()

eda_summary['split_ratio'] = {'train': TRAIN_RATIO, 'validation': VAL_RATIO, 'test': TEST_RATIO}
eda_summary['split_sizes'] = split_sizes
eda_summary['split_label_counts'] = split_counts
eda_summary['figures']['class_split_distribution'] = str(OUTPUT_DIR / 'eda_class_split_distribution.png')


# 02_preprocessing.ipynb


# Preprocessing dan Dataset Loader

Notebook ini membuat skenario preprocessing, menampilkan contoh hasil preprocessing, dan menyiapkan loader untuk model klasik serta CNN.

## 1. Enhancement, Restoration, dan Face Crop

CLAHE meningkatkan kontras lokal pada channel luminance LAB, Gaussian blur ringan mereduksi noise, dan face crop memakai Haar Cascade dengan fallback ke gambar penuh.

In [ ]:
def apply_clahe_rgb(image_rgb: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_l = clahe.apply(l_channel)
    enhanced_lab = cv2.merge((enhanced_l, a_channel, b_channel))
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2RGB)


def apply_gaussian_denoise(image_rgb: np.ndarray) -> np.ndarray:
    return cv2.GaussianBlur(image_rgb, ksize=(3, 3), sigmaX=0.5)


def crop_face_if_detected(image_rgb: np.ndarray, detector: cv2.CascadeClassifier) -> tuple[np.ndarray, bool]:
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(35, 35))
    if len(faces) == 0:
        return image_rgb, False

    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    pad = int(0.18 * max(w, h))
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(image_rgb.shape[1], x + w + pad)
    y2 = min(image_rgb.shape[0], y + h + pad)
    return image_rgb[y1:y2, x1:x2], True


def preprocess_image(
    image_path: Path,
    use_enhancement: bool = False,
    use_face_crop: bool = False,
    detector: cv2.CascadeClassifier | None = None,
) -> tuple[np.ndarray | None, bool]:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return None, False

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    face_detected = False
    if use_face_crop and detector is not None:
        image_rgb, face_detected = crop_face_if_detected(image_rgb, detector)
    if use_enhancement:
        image_rgb = apply_clahe_rgb(image_rgb)
        image_rgb = apply_gaussian_denoise(image_rgb)

    resized = cv2.resize(image_rgb, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    return resized, face_detected


def build_preprocessed_scenario(
    scenario_name: str,
    use_enhancement: bool = False,
    use_face_crop: bool = False,
) -> dict[str, int]:
    scenario_dir = SCENARIO_ROOT / scenario_name
    if scenario_dir.exists():
        shutil.rmtree(scenario_dir)

    cascade_path = Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'
    detector = cv2.CascadeClassifier(str(cascade_path)) if use_face_crop else None
    stats = {'processed': 0, 'read_failed': 0, 'faces_detected': 0}

    for split_name, records in splits.items():
        for idx, (image_path, class_name) in enumerate(records):
            target_class_dir = scenario_dir / split_name / class_name
            target_class_dir.mkdir(parents=True, exist_ok=True)

            image_rgb, face_detected = preprocess_image(
                image_path,
                use_enhancement=use_enhancement,
                use_face_crop=use_face_crop,
                detector=detector,
            )
            if image_rgb is None:
                stats['read_failed'] += 1
                continue

            output_path = target_class_dir / f'{idx:06d}_{image_path.stem}.jpg'
            cv2.imwrite(str(output_path), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))
            stats['processed'] += 1
            stats['faces_detected'] += int(face_detected)

    return stats


SCENARIO_CONFIGS = {
    'full_plain': {'use_enhancement': False, 'use_face_crop': False},
    'full_enhanced': {'use_enhancement': True, 'use_face_crop': False},
    'crop_enhanced': {'use_enhancement': True, 'use_face_crop': True},
}

preprocessing_stats = {}
for scenario_name, config in SCENARIO_CONFIGS.items():
    preprocessing_stats[scenario_name] = build_preprocessed_scenario(scenario_name, **config)

print('Preprocessing stats:', preprocessing_stats)
print_tree(SCENARIO_ROOT, max_depth=3, limit=90)

## 2. Visualisasi Dataset dan Preprocessing

Contoh gambar dari tiap skenario ditampilkan untuk memeriksa efek resize, enhancement, dan crop sebelum masuk ke tahap modeling.

In [ ]:
def read_rgb(path: Path) -> np.ndarray:
    image_bgr = cv2.imread(str(path))
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


preview_records = []
for class_name in CLASS_NAMES:
    preview_records.extend((path, class_name) for path in paths_by_class[class_name][:2])

cascade_path = Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'
preview_detector = cv2.CascadeClassifier(str(cascade_path))

plt.figure(figsize=(12, 10))
for row, (image_path, class_name) in enumerate(preview_records[:4]):
    original = read_rgb(image_path)
    plain, _ = preprocess_image(image_path, use_enhancement=False, use_face_crop=False)
    enhanced, _ = preprocess_image(image_path, use_enhancement=True, use_face_crop=False)
    crop_enhanced, detected = preprocess_image(
        image_path,
        use_enhancement=True,
        use_face_crop=True,
        detector=preview_detector,
    )
    images = [original, plain, enhanced, crop_enhanced]
    titles = [f'Original: {class_name}', 'Resize 128x128', 'CLAHE + Gaussian', f'Crop + enhance ({detected})']

    for col, (image, title) in enumerate(zip(images, titles)):
        plt.subplot(4, 4, row * 4 + col + 1)
        plt.imshow(image)
        plt.title(title)
        plt.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'preprocessing_examples.png', dpi=180)
plt.show()

plt.figure(figsize=(7, 4))
bars = plt.bar(class_counts.keys(), class_counts.values(), color=['#2f80ed', '#27ae60'])
plt.title('Jumlah Gambar per Class')
plt.xlabel('Class')
plt.ylabel('Jumlah gambar')
plt.bar_label(bars)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=160)
plt.show()

## 3. Dataset Loader, Normalisasi, dan Augmentation

Loader menyiapkan input konsisten untuk HOG-SVM dan CNN. Normalisasi 0-1 dipakai untuk CNN, sedangkan augmentation hanya diterapkan pada data train.

In [ ]:
geometry_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip('horizontal', seed=SEED),
        tf.keras.layers.RandomRotation(0.07, seed=SEED),
        tf.keras.layers.RandomZoom(0.10, seed=SEED),
        tf.keras.layers.RandomContrast(0.10, seed=SEED),
    ],
    name='realistic_geometry_augmentation',
)


def load_tf_split(scenario_name: str, split_name: str, shuffle: bool = False, batch_size: int = BATCH_SIZE) -> tf.data.Dataset:
    return tf.keras.utils.image_dataset_from_directory(
        SCENARIO_ROOT / scenario_name / split_name,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        color_mode='rgb',
        batch_size=batch_size,
        image_size=IMAGE_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )


def augment_images(images: tf.Tensor, labels: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    images = geometry_augmentation(images, training=True)
    images = tf.image.random_brightness(images, max_delta=18.0, seed=SEED)
    images = tf.clip_by_value(images, 0.0, 255.0)
    return images, labels


def normalize_images(images: tf.Tensor, labels: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    return tf.cast(images, tf.float32) / 255.0, tf.cast(labels, tf.float32)


def prepare_cnn_dataset(dataset: tf.data.Dataset, augment: bool = False) -> tf.data.Dataset:
    if augment:
        dataset = dataset.map(augment_images, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(normalize_images, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.cache()
    return dataset.prefetch(tf.data.AUTOTUNE)


def load_images_for_classical(scenario_name: str, split_name: str) -> tuple[np.ndarray, np.ndarray]:
    images = []
    labels = []
    for class_name in CLASS_NAMES:
        class_dir = SCENARIO_ROOT / scenario_name / split_name / class_name
        for image_path in sorted(class_dir.glob('*.jpg')):
            images.append(read_rgb(image_path))
            labels.append(CLASS_TO_LABEL[class_name])
    return np.array(images, dtype=np.uint8), np.array(labels, dtype=np.int64)

# 03_classical.ipynb


# Model Klasik HOG + SVM

Notebook ini hanya membangun model klasik berbasis fitur HOG dan Linear SVM. Evaluasi test set, confusion matrix, ROC, dan ringkasan metrik dipindahkan ke notebook evaluasi.

## 1. Ekstraksi Fitur HOG

Fitur HOG menangkap pola orientasi gradien dari tepi wajah dan masker. Fitur ini menjadi input untuk baseline Linear SVM.

In [ ]:

def extract_hog_features(images: np.ndarray) -> np.ndarray:
    hog = cv2.HOGDescriptor(
        (128, 128),
        (16, 16),
        (8, 8),
        (8, 8),
        9,
    )
    features = []
    for image in images:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        descriptor = hog.compute(gray).flatten()
        features.append(descriptor)
    return np.array(features, dtype=np.float32)


## 2. Training Model SVM

Dua baseline dilatih pada gambar full plain dan full enhanced. Model disimpan dalam dictionary agar evaluasi dilakukan terpusat di notebook evaluasi.

In [ ]:

import joblib
classical_experiments = [
    {
        'scenario_id': 'A_hog_svm_plain_full',
        'scenario_name': 'full_plain',
        'config': {'feature': 'HOG', 'classifier': 'LinearSVC', 'enhancement': False},
    },
    {
        'scenario_id': 'B_hog_svm_enhanced_full',
        'scenario_name': 'full_enhanced',
        'config': {'feature': 'HOG', 'classifier': 'LinearSVC', 'enhancement': True},
    },
]

classical_models = {}
classical_feature_sets = {}

for experiment in classical_experiments:
    scenario_id = experiment['scenario_id']
    scenario_name = experiment['scenario_name']

    X_train_img, y_train = load_images_for_classical(scenario_name, 'train')
    X_val_img, y_val = load_images_for_classical(scenario_name, 'validation')

    X_train = extract_hog_features(X_train_img)
    X_val = extract_hog_features(X_val_img)

    svm_model = make_pipeline(
        StandardScaler(),
        LinearSVC(C=1.0, class_weight='balanced', max_iter=8000, random_state=SEED),
    )
    svm_model.fit(X_train, y_train)
    val_pred = svm_model.predict(X_val)
    val_accuracy = float(accuracy_score(y_val, val_pred))

    model_path = OUTPUT_DIR / f'model_{scenario_id}.joblib'
    joblib.dump(svm_model, model_path)

    classical_models[scenario_id] = svm_model
    classical_feature_sets[scenario_id] = {
        'scenario_name': scenario_name,
        'config': experiment['config'],
        'validation_accuracy': val_accuracy,
        'model_path': str(model_path),
    }

    print(f'Trained {scenario_id}')
    print(f'Validation accuracy: {val_accuracy:.4f}')
    print(f'Saved model: {model_path}')


# 04_cnn.ipynb


# Model CNN From Scratch

Notebook ini hanya mendefinisikan arsitektur CNN, menjalankan training, dan menyimpan model serta history. Evaluasi test set dan visualisasi evaluasi dipindahkan ke notebook evaluasi.

## 1. Arsitektur Custom CNN

Model dibuat dari layer Conv2D, BatchNormalization, ReLU, Dropout, GlobalAveragePooling, Dense, dan optional Squeeze-and-Excitation block tanpa pretrained weights.

In [ ]:
def make_optimizer(name: str, learning_rate: float):
    name = name.lower()
    if name == 'adamw' and hasattr(tf.keras.optimizers, 'AdamW'):
        return tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4)
    return tf.keras.optimizers.Adam(learning_rate=learning_rate)


def conv_bn_relu(x: tf.Tensor, filters: int) -> tf.Tensor:
    x = tf.keras.layers.Conv2D(filters, kernel_size=3, padding='same', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    return tf.keras.layers.ReLU()(x)


def squeeze_excite_block(x: tf.Tensor, ratio: int = 8) -> tf.Tensor:
    channels = int(x.shape[-1])
    se = tf.keras.layers.GlobalAveragePooling2D()(x)
    se = tf.keras.layers.Dense(max(channels // ratio, 8), activation='relu')(se)
    se = tf.keras.layers.Dense(channels, activation='sigmoid')(se)
    se = tf.keras.layers.Reshape((1, 1, channels))(se)
    return tf.keras.layers.Multiply()([x, se])


def build_custom_cnn(config: dict) -> tf.keras.Model:
    filters = config.get('filters', [32, 64, 128, 256])
    dropout_scale = float(config.get('dropout_scale', 1.0))
    use_se = bool(config.get('use_se', True))
    dense_units = int(config.get('dense_units', 128))
    optimizer_name = config.get('optimizer', 'adamw')
    learning_rate = float(config.get('learning_rate', 1e-3))

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = inputs
    block_dropouts = [0.15, 0.20, 0.25, 0.30]

    for block_idx, filter_count in enumerate(filters):
        x = conv_bn_relu(x, filter_count)
        x = conv_bn_relu(x, filter_count)
        if use_se and block_idx in [2, 3]:
            x = squeeze_excite_block(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Dropout(block_dropouts[block_idx] * dropout_scale)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(dense_units, activation='relu', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.40 * dropout_scale)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs, name='custom_cnn_from_scratch')
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
    )
    return model


base_cnn_config = {
    'filters': [32, 64, 128, 256],
    'dropout_scale': 1.0,
    'use_se': True,
    'dense_units': 128,
    'optimizer': 'adamw',
    'learning_rate': 1e-3,
    'batch_size': 32,
}

build_custom_cnn(base_cnn_config).summary()

## 2. Training CNN

Fungsi training mengembalikan model, checkpoint, history, dan konfigurasi. Prediksi test set dihitung nanti pada notebook evaluasi.

In [ ]:

def train_cnn_model(scenario_id: str, scenario_name: str, config: dict, augment_train: bool = False) -> tuple[dict, tf.keras.Model]:
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    batch_size = int(config.get('batch_size', BATCH_SIZE))
    train_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'train', shuffle=True, batch_size=batch_size), augment=augment_train)
    val_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'validation', shuffle=False, batch_size=batch_size))

    model = build_custom_cnn(config)
    checkpoint_path = OUTPUT_DIR / f'best_{scenario_id}.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
            mode='min',
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.4,
            patience=LR_PATIENCE,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True,
        ),
    ]

    print(f'\nTraining {scenario_id}')
    print('Config:', config, 'augment_train:', augment_train)
    history_obj = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    history = history_obj.history

    record = {
        'scenario': scenario_id,
        'scenario_name': scenario_name,
        'model_type': 'Custom CNN Conv2D from scratch',
        'config': {**config, 'augmentation': bool(augment_train), 'scenario_dataset': scenario_name},
        'history': {key: [float(v) for v in values] for key, values in history.items()},
        'checkpoint_path': str(checkpoint_path),
        'epochs_trained': len(history.get('loss', [])),
        'best_val_loss': float(min(history.get('val_loss', [np.inf]))),
        'best_val_accuracy': float(max(history.get('val_accuracy', [0.0]))),
    }
    return record, model


## 3. Skenario Training CNN

Skenario training mencakup plain/enhanced, crop, augmentation, SE ablation, dan tuning ringan. Semua hasil mentah disimpan untuk evaluasi berikutnya.

In [ ]:
MAX_EPOCHS = 50
print(f'CNN max epochs: {MAX_EPOCHS} | early stopping patience: {EARLY_STOP_PATIENCE}')

cnn_experiments = [
    {
        'scenario_id': 'C_cnn_plain_full_se_no_aug',
        'scenario_name': 'full_plain',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'CNN tanpa enhancement',
    },
    {
        'scenario_id': 'D_cnn_enhanced_full_se_no_aug',
        'scenario_name': 'full_enhanced',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh enhancement',
    },
    {
        'scenario_id': 'F_cnn_enhanced_full_se_aug',
        'scenario_name': 'full_enhanced',
        'augment': True,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh augmentation',
    },
]

if RUN_OPTIONAL_CROP_SCENARIO:
    cnn_experiments.append({
        'scenario_id': 'E_cnn_enhanced_crop_se_no_aug',
        'scenario_name': 'crop_enhanced',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh face crop',
    })

if RUN_SE_ABLATION:
    cnn_experiments.append({
        'scenario_id': 'G_cnn_enhanced_full_no_se_aug',
        'scenario_name': 'full_enhanced',
        'augment': True,
        'config': {**base_cnn_config, 'use_se': False},
        'purpose': 'Ablation SE block',
    })

if RUN_LIGHT_TUNING:
    cnn_experiments.extend([
        {
            'scenario_id': 'T1_cnn_enhanced_full_se_aug_lr5e4',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'learning_rate': 5e-4, 'dropout_scale': 1.0, 'optimizer': 'adamw'},
            'purpose': 'Tuning learning rate',
        },
        {
            'scenario_id': 'T2_cnn_enhanced_full_se_aug_wider',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'filters': [48, 96, 160, 256], 'learning_rate': 7e-4, 'dropout_scale': 1.1, 'optimizer': 'adamw'},
            'purpose': 'Tuning jumlah filter dan dropout',
        },
        {
            'scenario_id': 'T3_cnn_enhanced_full_se_aug_adam',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'optimizer': 'adam', 'learning_rate': 1e-3, 'dropout_scale': 0.9, 'batch_size': 64},
            'purpose': 'Tuning optimizer dan batch size',
        },
    ])

trained_models = {}
cnn_training_records = {}
for experiment in cnn_experiments:
    record, model = train_cnn_model(
        experiment['scenario_id'],
        experiment['scenario_name'],
        experiment['config'],
        augment_train=experiment['augment'],
    )
    record['purpose'] = experiment['purpose']
    cnn_training_records[experiment['scenario_id']] = record
    trained_models[experiment['scenario_id']] = model


# 05_evaluation.ipynb


# Evaluasi dan Export Artefak

Notebook ini menghitung evaluasi untuk seluruh model klasik dan CNN, membuat visualisasi evaluasi, memilih model final, lalu menyimpan metrics dan model terbaik.

## 1. Fungsi Evaluasi

Confusion matrix, ROC curve, classification report, training curve, dan tabel ringkasan dibuat terpusat agar notebook modeling tetap fokus pada training model.

In [ ]:

def plot_confusion_matrix(matrix: np.ndarray, scenario_id: str) -> None:
    fig, ax = plt.subplots(figsize=(5, 5))
    display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=CLASS_NAMES)
    display.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    plt.title(f'Confusion Matrix - {scenario_id}')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'confusion_matrix_{scenario_id}.png', dpi=180)
    plt.show()


def plot_roc_curve(y_true: np.ndarray, y_score: np.ndarray, scenario_id: str) -> float:
    plt.figure(figsize=(5.5, 5))
    if len(np.unique(y_true)) < 2:
        roc_auc = float('nan')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='AUC = nan')
        plt.text(0.5, 0.5, 'ROC tidak dihitung\nkarena hanya ada satu kelas', ha='center', va='center')
    else:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {scenario_id}')
    plt.grid(alpha=0.3)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'roc_curve_{scenario_id}.png', dpi=180)
    plt.show()
    return float(roc_auc)


def plot_training_history(history: dict, scenario_id: str) -> None:
    plt.figure(figsize=(11, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.get('accuracy', []), marker='o', label='train')
    plt.plot(history.get('val_accuracy', []), marker='o', label='validation')
    plt.title(f'Accuracy - {scenario_id}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.get('loss', []), marker='o', label='train')
    plt.plot(history.get('val_loss', []), marker='o', label='validation')
    plt.title(f'Loss - {scenario_id}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'training_history_{scenario_id}.png', dpi=180)
    plt.show()


def evaluate_predictions(
    scenario_id: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: np.ndarray,
    model_type: str,
    config: dict | None = None,
    history: dict | None = None,
    test_loss: float | None = None,
    extra: dict | None = None,
) -> dict:
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    plot_confusion_matrix(matrix, scenario_id)
    roc_auc = plot_roc_curve(y_true, y_score, scenario_id)

    result = {
        'scenario': scenario_id,
        'model_type': model_type,
        'config': config or {},
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_true, y_pred, zero_division=0)),
        'auc': roc_auc,
        'classification_report': report_dict,
        'confusion_matrix': matrix.tolist(),
    }
    if history is not None:
        result['history'] = {key: [float(v) for v in values] for key, values in history.items()}
    if test_loss is not None:
        result['test_loss'] = float(test_loss)
    if extra:
        result.update(extra)

    print(f'\n=== {scenario_id} ===')
    print(f"Accuracy : {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall   : {result['recall']:.4f}")
    print(f"F1-score : {result['f1_score']:.4f}")
    print(f"AUC      : {result['auc']:.4f}")
    print(classification_report(y_true, y_pred, labels=[0, 1], target_names=CLASS_NAMES, zero_division=0))
    if history is not None:
        plot_training_history(history, scenario_id)
    return result



## 2. Evaluasi Model Klasik

Model SVM yang sudah dilatih dievaluasi pada test set menggunakan HOG feature extraction yang sama dengan tahap training.

In [ ]:

scenario_results = []

for experiment in classical_experiments:
    scenario_id = experiment['scenario_id']
    scenario_name = experiment['scenario_name']
    model = classical_models[scenario_id]

    X_test_img, y_test = load_images_for_classical(scenario_name, 'test')
    X_test = extract_hog_features(X_test_img)
    y_score = model.decision_function(X_test)
    y_pred = (y_score >= 0).astype(np.int64)

    scenario_results.append(
        evaluate_predictions(
            scenario_id,
            y_test,
            y_pred,
            y_score,
            model_type='HOG + LinearSVC',
            config=experiment['config'],
            extra={
                'validation_accuracy': classical_feature_sets[scenario_id]['validation_accuracy'],
                'model_path': classical_feature_sets[scenario_id].get('model_path'),
            },
        )
    )


## 3. Evaluasi Model CNN

Setiap model CNN dievaluasi pada test set sesuai skenario dataset-nya. Prediksi, metric Keras, training history, dan checkpoint dicatat pada hasil evaluasi.

In [ ]:

for scenario_id, record in cnn_training_records.items():
    model = trained_models[scenario_id]
    config = record['config']
    scenario_name = record['scenario_name']
    batch_size = int(config.get('batch_size', BATCH_SIZE))
    test_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'test', shuffle=False, batch_size=batch_size))

    test_metrics = model.evaluate(test_ds, verbose=0)
    metric_names = model.metrics_names
    test_metric_dict = {name: float(value) for name, value in zip(metric_names, test_metrics)}

    y_true = []
    y_score = []
    for images, labels in test_ds:
        scores = model.predict(images, verbose=0).reshape(-1)
        y_true.extend(labels.numpy().reshape(-1).astype(np.int64).tolist())
        y_score.extend(scores.tolist())

    y_true = np.array(y_true, dtype=np.int64)
    y_score = np.array(y_score, dtype=np.float32)
    y_pred = (y_score >= 0.5).astype(np.int64)

    result = evaluate_predictions(
        scenario_id,
        y_true,
        y_pred,
        y_score,
        model_type='Custom CNN Conv2D from scratch',
        config=config,
        history=record['history'],
        test_loss=test_metric_dict.get('loss'),
        extra={
            'keras_test_metrics': test_metric_dict,
            'checkpoint_path': record['checkpoint_path'],
            'epochs_trained': record['epochs_trained'],
            'best_val_loss': record['best_val_loss'],
            'best_val_accuracy': record['best_val_accuracy'],
            'purpose': record.get('purpose', ''),
        },
    )
    scenario_results.append(result)


## Sampel Prediksi dengan Bounding Box

Dua contoh gambar dari test set ditampilkan untuk setiap model: satu `with_mask` dan satu `without_mask`. Gambar asli diberi bounding box wajah terbesar, lalu judul panel menampilkan label asli dan hasil prediksi model agar keluaran tiap skenario dapat dibandingkan secara visual.


In [ ]:
def detect_largest_face_box(image_rgb: np.ndarray, detector: cv2.CascadeClassifier) -> tuple[tuple[int, int, int, int], bool]:
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(35, 35))
    if len(faces) == 0:
        return (0, 0, image_rgb.shape[1], image_rgb.shape[0]), False
    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    return (int(x), int(y), int(w), int(h)), True


def draw_prediction_bbox(image_rgb: np.ndarray, box: tuple[int, int, int, int], label: str, detected: bool) -> np.ndarray:
    canvas = image_rgb.copy()
    x, y, w, h = box
    color = (39, 174, 96) if label == 'with_mask' else (235, 87, 87)
    line_width = max(2, min(4, image_rgb.shape[1] // 180))
    cv2.rectangle(canvas, (x, y), (x + w, y + h), color, line_width)

    # Fallback full-image box is only a visual boundary; avoid long text that covers the face.
    text = label if detected else f'{label} (fallback)'
    font_scale = max(0.35, min(0.65, image_rgb.shape[1] / 720))
    thickness = max(1, min(2, image_rgb.shape[1] // 360))
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)

    text_x = max(2, min(x, canvas.shape[1] - tw - 8))
    if y - th - baseline - 8 >= 0:
        text_y = y - 6
        bg_y1 = text_y - th - baseline - 4
        bg_y2 = text_y + baseline + 2
    else:
        text_y = min(canvas.shape[0] - baseline - 4, y + h + th + 8)
        bg_y1 = text_y - th - baseline - 4
        bg_y2 = text_y + baseline + 2

    cv2.rectangle(canvas, (text_x, max(0, bg_y1)), (min(text_x + tw + 8, canvas.shape[1]), min(bg_y2, canvas.shape[0])), color, -1)
    cv2.putText(canvas, text, (text_x + 4, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
    return canvas


def get_test_sample_by_class(class_name: str) -> Path:
    fallback_path = None
    for image_path, label in splits['test']:
        if label != class_name:
            continue
        if fallback_path is None:
            fallback_path = image_path
        image_rgb = load_original_rgb(image_path)
        _, detected = detect_largest_face_box(image_rgb, sample_bbox_detector)
        if detected:
            return image_path
    if fallback_path is None:
        raise ValueError(f'Tidak ada sampel test untuk class {class_name}')
    return fallback_path


def predict_sample_for_experiment(experiment: dict, image_path: Path, model_kind: str) -> str:
    scenario_id = experiment['scenario_id']
    scenario_name = experiment['scenario_name']
    use_enhancement = 'enhanced' in scenario_name
    use_face_crop = 'crop' in scenario_name
    processed, _ = preprocess_image(
        image_path,
        use_enhancement=use_enhancement,
        use_face_crop=use_face_crop,
        detector=sample_bbox_detector,
    )
    if processed is None:
        return 'unreadable'

    if model_kind == 'classical':
        features = extract_hog_features(np.array([processed], dtype=np.uint8))
        score = float(classical_models[scenario_id].decision_function(features).reshape(-1)[0])
        pred_idx = int(score >= 0.0)
    else:
        model_input = np.expand_dims(processed.astype(np.float32) / 255.0, axis=0)
        score = float(trained_models[scenario_id].predict(model_input, verbose=0).reshape(-1)[0])
        pred_idx = int(score >= 0.5)
    return CLASS_NAMES[pred_idx]


def short_scenario_name(scenario_id: str) -> str:
    return scenario_id.replace('_cnn_', '_cnn\n').replace('_hog_', '_hog\n')


def plot_sample_predictions_with_bbox() -> None:
    sample_paths = {class_name: get_test_sample_by_class(class_name) for class_name in CLASS_NAMES}
    print('Sample paths used for bbox visualization:')
    for class_name, image_path in sample_paths.items():
        print(f'- {class_name}: {image_path}')

    experiments = [(experiment, 'classical') for experiment in classical_experiments]
    experiments.extend((experiment, 'cnn') for experiment in cnn_experiments)

    fig, axes = plt.subplots(len(experiments), len(CLASS_NAMES), figsize=(11, max(3, len(experiments) * 2.7)))
    if len(experiments) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, (experiment, model_kind) in enumerate(experiments):
        for col_idx, class_name in enumerate(CLASS_NAMES):
            image_path = sample_paths[class_name]
            original = load_original_rgb(image_path)
            pred_label = predict_sample_for_experiment(experiment, image_path, model_kind)
            box, detected = detect_largest_face_box(original, sample_bbox_detector)
            visual = draw_prediction_bbox(original, box, pred_label, detected)

            ax = axes[row_idx, col_idx]
            ax.imshow(visual)
            ax.set_title(f"{short_scenario_name(experiment['scenario_id'])}\ntrue: {class_name} | pred: {pred_label}", fontsize=8)
            ax.axis('off')

    plt.tight_layout(pad=1.1)
    output_path = OUTPUT_DIR / 'sample_predictions_with_bbox_by_model.png'
    plt.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved sample prediction grid: {output_path}')


sample_bbox_detector = cv2.CascadeClassifier(str(Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'))
plot_sample_predictions_with_bbox()


## 4. Perbandingan Semua Skenario

Seluruh hasil klasik dan CNN diringkas dalam tabel dan grafik perbandingan agar scenario terbaik terlihat jelas.

In [ ]:

def plot_scenario_comparison(results: list[dict]) -> None:
    names = [result['scenario'] for result in results]
    accuracy = [result['accuracy'] for result in results]
    precision = [result['precision'] for result in results]
    recall = [result['recall'] for result in results]
    f1 = [result['f1_score'] for result in results]
    auc_scores = [result['auc'] for result in results]

    x = np.arange(len(names))
    width = 0.16
    plt.figure(figsize=(16, 6))
    plt.bar(x - 2 * width, accuracy, width, label='accuracy')
    plt.bar(x - width, precision, width, label='precision')
    plt.bar(x, recall, width, label='recall')
    plt.bar(x + width, f1, width, label='f1-score')
    plt.bar(x + 2 * width, auc_scores, width, label='AUC')
    plt.xticks(x, names, rotation=30, ha='right')
    plt.ylim(0, 1.05)
    plt.ylabel('Score')
    plt.title('Perbandingan Semua Skenario')
    plt.grid(axis='y', alpha=0.3)
    plt.legend(ncol=5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scenario_comparison.png', dpi=180)
    plt.show()


def summarize_results(results: list[dict]) -> list[dict]:
    rows = []
    for result in results:
        rows.append({
            'scenario': result['scenario'],
            'model_type': result['model_type'],
            'accuracy': round(result['accuracy'], 4),
            'precision': round(result['precision'], 4),
            'recall': round(result['recall'], 4),
            'f1_score': round(result['f1_score'], 4),
            'auc': round(result['auc'], 4),
            'epochs': result.get('epochs_trained'),
            'best_val_loss': None if result.get('best_val_loss') is None else round(result['best_val_loss'], 4),
            'best_val_accuracy': None if result.get('best_val_accuracy') is None else round(result['best_val_accuracy'], 4),
            'purpose': result.get('purpose', ''),
            'config': result.get('config', {}),
        })
    return rows


summary_rows = summarize_results(scenario_results)
print(json.dumps(summary_rows, indent=2))
plot_scenario_comparison(scenario_results)

best_test_result = max(scenario_results, key=lambda item: (item['f1_score'], item['auc'], item['accuracy']))
cnn_results = [result for result in scenario_results if result['model_type'] == 'Custom CNN Conv2D from scratch']
best_cnn_result = min(cnn_results, key=lambda item: item.get('best_val_loss', np.inf))
print('Best test scenario for reporting:', best_test_result['scenario'])
print('Final selected CNN scenario by validation loss:', best_cnn_result['scenario'])
print('Final selected CNN config:', json.dumps(best_cnn_result['config'], indent=2))


## 5. Analisis Akademik

Dataset dipilih karena sesuai langsung dengan masalah klasifikasi penggunaan masker dan memiliki dua kelas yang jelas. Karakteristik dataset meliputi variasi pose, jarak wajah, background, pencahayaan, kualitas gambar, dan kemungkinan oklusi selain masker. Variasi ini membuat pipeline computer vision perlu diuji dari preprocessing hingga evaluasi akhir.

Masalah pencahayaan dan noise ditangani dengan enhancement/restoration ringan. CLAHE meningkatkan kontras lokal pada channel luminance sehingga detail area wajah dan masker dapat lebih terlihat, sedangkan Gaussian denoise ringan mengurangi noise tanpa merusak bentuk masker. Face crop bertujuan mengurangi pengaruh background, tetapi tetap memiliki risiko jika Haar Cascade gagal mendeteksi wajah pada pose ekstrem atau citra blur.

HOG digunakan sebagai baseline klasik karena menangkap distribusi orientasi tepi dan bentuk lokal. Baseline ini penting untuk menunjukkan apakah fitur manual masih kompetitif dibanding fitur yang dipelajari otomatis. CNN from scratch digunakan karena requirement melarang bobot eksternal; seluruh representasi dipelajari langsung dari dataset praktikum.

SE block digunakan sebagai attention channel ringan. Tujuannya adalah membantu model memberi bobot lebih besar pada channel fitur yang relevan tanpa menambah model eksternal. Ablation tanpa SE diperlukan untuk melihat apakah attention benar-benar membantu atau justru menambah kompleksitas.

Augmentation realistis membantu generalisasi terhadap variasi kecil seperti arah wajah, zoom, kontras, dan brightness. Augmentation ekstrem dihindari karena dapat mengubah bentuk masker atau membuat citra tidak realistis. Overfitting dianalisis dari jarak antara training dan validation accuracy/loss. Jika training accuracy tinggi tetapi validation loss naik, model mulai menghafal data. ReduceLROnPlateau, Dropout, BatchNormalization, dan EarlyStopping digunakan untuk menekan risiko tersebut.

Kekuatan sistem ini adalah pipeline lengkap, skenario jelas, dan evaluasi menyeluruh. Kelemahannya adalah Haar Cascade tidak selalu robust, tuning masih ringan, dan performa sangat bergantung pada kualitas/variasi dataset. Pengembangan lanjutan dapat mencakup cross-validation, threshold tuning berbasis validation set, analisis error per kondisi citra, balancing tambahan jika class tidak seimbang, dan face detector yang lebih stabil selama tetap sesuai aturan praktikum.

## 6. Simpan Artefak Final

Model CNN final dipilih berdasarkan validation loss terbaik dari skenario CNN yang dijalankan. Test set tetap dipakai untuk evaluasi akhir dan pelaporan, bukan untuk memilih model.

In [ ]:
best_model = trained_models[best_cnn_result['scenario']]
final_keras_path = OUTPUT_DIR / 'face_mask_custom_cnn_from_scratch_best.keras'
metrics_path = OUTPUT_DIR / 'metrics.json'
best_model.save(final_keras_path)

metrics = {
    'seed': SEED,
    'tensorflow_version': tf.__version__,
    'dataset_handle': DATASET_HANDLE,
    'dataset_root': str(dataset_root),
    'original_image_directory': str(original_image_dir),
    'class_names': CLASS_NAMES,
    'class_counts': class_counts,
    'split_ratio': {'train': TRAIN_RATIO, 'validation': VAL_RATIO, 'test': TEST_RATIO},
    'split_sizes': split_sizes,
    'split_label_counts': split_counts,
    'image_size': list(IMAGE_SIZE),
    'batch_size_default': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'eda_summary': eda_summary,
    'preprocessing_stats': preprocessing_stats,
    'deep_learning_model': 'Custom CNN Conv2D from scratch',
    'best_test_scenario_for_reporting': best_test_result['scenario'],
    'best_cnn_scenario': best_cnn_result['scenario'],
    'best_cnn_config': best_cnn_result['config'],
    'summary_rows': summary_rows,
    'scenario_results': scenario_results,
    'outputs': {
        'best_keras_model': str(final_keras_path),
        'metrics': str(metrics_path),
        'classical_models': {
            scenario_id: classical_feature_sets[scenario_id].get('model_path')
            for scenario_id in classical_feature_sets
        },
        'eda_class_distribution': str(OUTPUT_DIR / 'eda_class_distribution.png'),
        'eda_original_samples': str(OUTPUT_DIR / 'eda_original_samples.png'),
        'eda_resolution_aspect_ratio': str(OUTPUT_DIR / 'eda_resolution_aspect_ratio.png'),
        'eda_brightness_contrast': str(OUTPUT_DIR / 'eda_brightness_contrast.png'),
        'eda_noise_blur': str(OUTPUT_DIR / 'eda_noise_blur.png'),
        'eda_edge_hog_examples': str(OUTPUT_DIR / 'eda_edge_hog_examples.png'),
        'eda_pose_background_variation': str(OUTPUT_DIR / 'eda_pose_background_variation.png'),
        'eda_face_detection_coverage': str(OUTPUT_DIR / 'eda_face_detection_coverage.png'),
        'eda_face_detection_examples': str(OUTPUT_DIR / 'eda_face_detection_examples.png'),
        'eda_class_split_distribution': str(OUTPUT_DIR / 'eda_class_split_distribution.png'),
        'class_distribution': str(OUTPUT_DIR / 'class_distribution.png'),
        'preprocessing_examples': str(OUTPUT_DIR / 'preprocessing_examples.png'),
        'scenario_comparison': str(OUTPUT_DIR / 'scenario_comparison.png'),
    },
}
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

print('Saved outputs:')
for path in [
    final_keras_path,
    metrics_path,
    *[Path(path) for path in metrics['outputs']['classical_models'].values() if path],
    OUTPUT_DIR / 'eda_class_distribution.png',
    OUTPUT_DIR / 'eda_original_samples.png',
    OUTPUT_DIR / 'eda_resolution_aspect_ratio.png',
    OUTPUT_DIR / 'eda_brightness_contrast.png',
    OUTPUT_DIR / 'eda_noise_blur.png',
    OUTPUT_DIR / 'eda_edge_hog_examples.png',
    OUTPUT_DIR / 'eda_pose_background_variation.png',
    OUTPUT_DIR / 'eda_face_detection_coverage.png',
    OUTPUT_DIR / 'eda_face_detection_examples.png',
    OUTPUT_DIR / 'eda_class_split_distribution.png',
    OUTPUT_DIR / 'class_distribution.png',
    OUTPUT_DIR / 'preprocessing_examples.png',
    OUTPUT_DIR / 'scenario_comparison.png',
]:
    if path.exists():
        print('-', path)

In [ ]:
# Zip hasil agar mudah diunduh dari Colab.
import shutil
zip_base = Path('/content/face_mask_results')
zip_path = shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
print('Results zip:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as error:
    print('Download otomatis dilewati:', repr(error))
